# 01 — Build Training Dataset

This notebook constructs the quasar catalog used throughout the pipeline.

**Input:**  `data/raw/dr16q_prop_May01_2024.fits` — SDSS DR16Q quasar properties catalog  
**Outputs:**
- `data/raw/DR16Q_z_le2_with_mass.fits` — redshift- and mass-filtered quasars  
- `data/DR16Q_final.fits` — adds SDSS photometry, keeps bright sources (r < 20)  
- `data/DR16Q_final_stripe82.fits` — Stripe 82 sky footprint subset (used for ZTF matching)

**Reference catalog:** [Wu & Shen 2022, ApJS](https://iopscience.iop.org/article/10.3847/1538-4365/ac9ead/pdf)  
Downloaded from: http://quasar.astro.illinois.edu/paper_data/DR16Q/  
Original DR16Q: https://data.sdss.org/datamodel/files/BOSS_QSO/DR16Q/DR16Q_v4.html


In [ ]:
# Install required packages (uncomment if running for the first time)
# !pip install astroquery astropy pandas pyarrow tqdm

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm

from astropy.io import fits
from astropy.table import Table, join
from astropy.coordinates import SkyCoord
import astropy.units as u

# ── Directory layout ──────────────────────────────────────────────────────────
DATA_DIR = Path("data")
RAW_DIR  = DATA_DIR / "raw"
LC_DIR   = RAW_DIR / "ztf_lightcurves"

DATA_DIR.mkdir(parents=True, exist_ok=True)
LC_DIR.mkdir(parents=True, exist_ok=True)


## 1. Inspect the DR16Q Catalog

Peek at column names and confirm that black-hole mass columns (*`MBH`*) are present.


In [ ]:
DR16_PATH = RAW_DIR / "dr16q_prop_May01_2024.fits"

with fits.open(DR16_PATH, memmap=True) as hdul:
    cols = hdul[1].columns
    print("All columns:", cols.names)
    print("MBH columns:", [name for name in cols.names if "MBH" in name])
    print(f"Total quasars in catalog: {hdul[1].header['NAXIS2']:,}")


## 2. Filter: Redshift ≤ 2 and Valid Black-Hole Mass

We restrict to z ≤ 2 so that the Hα broad line (the preferred virial mass estimator)
falls within optical wavelengths, and require LOGMBH > 0 to exclude objects with
unreliable or missing mass measurements.


In [ ]:
t = Table.read(DR16_PATH)

# Keep physically usable quasars only
t_filtered = t[(t['Z_DR16Q'] <= 2) & (t['LOGMBH'] > 0)]

out_path = RAW_DIR / "DR16Q_z_le2_with_mass.fits"
t_filtered.write(out_path, overwrite=True)

print(f"Original rows : {len(t):,}")
print(f"Filtered rows : {len(t_filtered):,}")
print(f"Saved to      : {out_path}")


## 3. Add SDSS Photometry and Apply Brightness Cut

The mass-estimated catalog (`dr16q_prop_May01_2024.fits`) does not carry photometry,
so we cross-match back to the original `DR16Q_v4.fits` on `SDSS_NAME` to recover the
five-band PSF magnitudes (u, g, **r**, i, z).

We then keep only sources with r-band PSF magnitude < 20, which is the practical
brightness limit for reliable ZTF light curves.

| File                         | Description                                      |
|------------------------------|--------------------------------------------------|
| `DR16Q_v4.fits`              | Full raw DR16Q universe                          |
| `DR16Q_z_le2_with_mass.fits` | Physically usable subset (step 2)                |
| `DR16Q_final.fits`           | + photometry, r < 20 brightness cut              |


In [ ]:
t_filtered = Table.read(RAW_DIR / "DR16Q_z_le2_with_mass.fits")
t_original = Table.read(RAW_DIR / "DR16Q_v4.fits")

# Cross-match on SDSS_NAME to recover PSFMAG (u,g,r,i,z)
t_original_small = t_original[['SDSS_NAME', 'PSFMAG']]
t_joined = join(t_filtered, t_original_small, keys='SDSS_NAME', join_type='inner')

# PSFMAG[:,2] is the r-band magnitude
mag_r = t_joined['PSFMAG'][:, 2]

# Quick summary of the magnitude distribution
lt20       = np.sum(mag_r < 20)
btw20_22   = np.sum((mag_r >= 20) & (mag_r <= 22))
gt22       = np.sum(mag_r > 22)
print(f"r < 20  (bright, ZTF-usable): {lt20:,}")
print(f"20 ≤ r ≤ 22                 : {btw20_22:,}")
print(f"r > 22  (too faint)          : {gt22:,}")
print(f"Total checked               : {len(mag_r):,}")

# Apply brightness cut
t_final = t_joined[mag_r < 20]
print(f"\nFinal sample size (r < 20): {len(t_final):,}")

out_path = DATA_DIR / "DR16Q_final.fits"
t_final.write(out_path, overwrite=True)
print(f"Saved to: {out_path}")


## 4. Extract Stripe 82 Subset

Stripe 82 is the deep SDSS equatorial stripe that has been repeatedly observed,
making it ideal for variability studies. We select objects within its footprint:

- Declination: −1.5° to +1.5°  
- RA: 310° to 360° ∪ 0° to 60°


In [ ]:
t_final = Table.read(DATA_DIR / "DR16Q_final.fits")

ra  = t_final['RA']
dec = t_final['DEC']

in_stripe82 = (
    (dec > -1.5) & (dec < 1.5) &
    ((ra > 310) | (ra < 60))
)

t_stripe82 = t_final[in_stripe82]

out_path = DATA_DIR / "DR16Q_final_stripe82.fits"
t_stripe82.write(out_path, overwrite=True)

print(f"Objects in full sky sample : {len(t_final):,}")
print(f"Objects in Stripe 82       : {len(t_stripe82):,}")
print(f"Columns                    : {t_stripe82.colnames}")
print(f"Saved to                   : {out_path}")


## 5. Preview the Final Stripe 82 Catalog

`PSFMAG` is a 5-element array (u, g, r, i, z), so we handle it separately
when converting to a pandas DataFrame for display.


In [ ]:
import pandas as pd

t_stripe82 = Table.read(DATA_DIR / "DR16Q_final_stripe82.fits")

# Separate 1-D columns from multi-dimensional ones
scalar_cols = ['SDSS_NAME', 'RA', 'DEC', 'Z_DR16Q', 'LOGMBH']
preview_df  = t_stripe82[scalar_cols].to_pandas()
preview_df['PSFMAG (u,g,r,i,z)'] = [list(x) for x in t_stripe82['PSFMAG']]

with pd.option_context('display.max_columns', None, 'display.max_colwidth', None,
                       'display.width', None, 'display.precision', 4):
    print(preview_df.head(5).to_string(index=False))
